# Open Source, Local, Token Cost

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm import tqdm

## Connect to OpenRouter

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

if openrouter_api_key:
    print('OpenRouter API key loaded successfully,')
else:
    print('Failed to load OpenRouter API key')

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, 
                    api_key=openrouter_api_key,
                    timeout=120,
                    max_retries=5)

OpenRouter API key loaded successfully,


## Part 1: The Data And A Smaller Prompt

In [3]:
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files"""
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False,
                    held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id. 
    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """

    claims = load('evals/held_out_claims.json' if held_out else 'claims.json')
    decisions = load('evals/held_out_decisions.json' if held_out else 'evals/decisions.json')
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [4]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
claims = [claim for claim in claims if labels[claim['claim_id']]['difficulty'] != 'manual']
policies = {policy['policy_id']: policy for policy in load('policies.json')}
print(f"{len(claims)} claims")

20 claims


In [5]:
decisions = pd.DataFrame([labels[claim['claim_id']] for claim in claims])
decisions['decision'].value_counts()

decision
approve    12
decline     5
refer       3
Name: count, dtype: int64

### A Smaller Prompt

In [6]:
policy_wording= (DATA / 'policy_wording.md').read_text(encoding='utf-8')
short_wording = (DATA / 'policy_wording_short.md').read_text(encoding='utf-8')

print(f'The policy wording is {len(policy_wording.split()):,} words')
print(f'The short policy wording is {len(short_wording.split()):,} words')

The policy wording is 1,369 words
The short policy wording is 808 words


In [7]:
def split_sections(wording):
    """ Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "##" + part for part in parts[1:]}
    return parts[0], sections

opening, sections = split_sections(short_wording)
list(sections)

['Limits',
 'Cancellation and Interruption',
 'Trip Delay',
 'Baggage Delay',
 'Baggage',
 'Passport',
 'Emergency Medical',
 'Add-ons',
 'Never covered',
 'Payment',
 'Claims']

In [8]:
BENEFITS = {
    'trip_cancellation': 'Cancellation and Interruption',
    'trip_interruption': 'Cancellation and Interruption',
    'trip_delay': 'Trip Delay',
    'baggage_delay': 'Baggage Delay',
    'baggage': 'Baggage',
    'passport': 'Passport',
    'emergency_medical': 'Emergency Medical' 
}

def make_wording(claim):
    """ The sections of the short wording that this claim needs: the limits, 
    its benefits, the add-ons if the policy has any, and the rules for  every claim.
    """
    headings = ['Limits', BENEFITS[claim['claim_type']]]
    if policies[claim['policy_id']]['add_ons']:
        headings.append('Add-ons')
    headings += ['Never covered', 'Payment', 'Claims']
    return '\n\n'.join([opening.strip()] + [sections[heading].strip()for heading in headings])

In [9]:
INSTRUCTIONS = """ You are a claims handler at UnderCover Travel Insurance.
Decide the claim uou are given, using the policy wording below and the customer's 
policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.
"""

def make_user_prompt(claim):
    """ The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim['policy_id']]
    record = {key: value for key, value in policy.items() if key not in ['policy_id', 'customer_id', 'sold_through', 'status']}
    details = {key: value for key, value in claim.items() if key not in ['policy_id', 'customer_id', 'status']}
    details['documents'] = [{'name': document['name'], 'type': document['type']} for document in claim['documents']]
    return f""" # Policy record
    
{json.dumps(record)}

# Claim

{json.dumps(details)}
"""

def make_messages(claim):
    """ Build the messages for one claim: the task and the the wording it needs,
    then the policy record and the claim.
    """
    return [
        {
            'role': 'system',
            'content': f"{INSTRUCTIONS}\n{make_wording(claim)}\n"
        },
        {
            'role': 'user',
            'content': make_user_prompt(claim)
        },
    ]

In [10]:
messages = make_messages(claims[0])
print(messages[0]['content'])
print(messages[1]['content'])

 You are a claims handler at UnderCover Travel Insurance.
Decide the claim uou are given, using the policy wording below and the customer's 
policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.

# UnderCover Travel Insurance: Policy Wording, short version

All amounts are in US dollars. The policy record shows the plan, the add-ons, the trip dates, the trip cost and the travelers.

##Limits

| Benefit | Economy | Premium | First |
|---|---|---|---|
| Cancellation and Interruption maximum | 1,500 | 5,000 | 10,000 |
| Trip Delay waiting period, hours | 12 | 6 | 3 |
| Trip Delay maximum | 300 | 800 | 1,500 |
| Baggage Delay waiting period, hours | 24 | 12 | 6 |
| Baggage Delay ma

## Part 2: Six Open-Weight Models on OpenRouter

In [11]:
class ClaimDecision(BaseModel):
    """ A class handler's decision on one class."""

    rationale: str= Field(description='2-3 sentences to explain the decision along with the relevant section of the policy')
    decision: Literal['approve', 
                      'decline',
                      'refer']
    payout: float = Field(description='The amount to pay in US dollars, or 0 for a decline or a referral')

In [80]:
class LocalDecision(ClaimDecision):
    """ ClaimDecision for Ollama, where a missing payout counts as 0, Gemma 4 sometimes leaves it out on declines"""
    payout: float = Field(default=0.0, description= 'The amount to pay in US Dollars, or 0 for a decline or a referral')

### Calling the LLM

In [81]:
def decide(claim, model, effort, ignore=None):
    """ Ask one model on OpenRouter to decide one claim, skipping any providers
    listed in ignore.
    """
    start = time.time()
    provider = {'require_parameters': True}

    if ignore:
        provider['ignore'] = ignore
    response = openrouter.chat.completions.parse(
        model = model,
        messages = make_messages(claim),
        response_format = ClaimDecision,
        reasoning_effort = effort,
        extra_body = {'provider': provider}
    )
    answer = response.choices[0].message.parsed
    decision, payout, rationale = answer.decision, answer.payout, answer.rationale
    return{
        'model': model,
        'effort': effort,
        'where': 'OpenRouter',
        'claim_id': claim['claim_id'],
        'decision': decision,
        'payout': payout,
        'rationale': rationale,
        'provider': response.provider,
        'input_tokens': response.usage.prompt_tokens,
        'cost': response.usage.cost,
        'seconds': round(time.time() - start, 1)
    }

In [82]:
first = decide(claims[0], 
               'deepseek/deepseek-v4.1-flash',
               'low')
first

{'model': 'deepseek/deepseek-v4.1-flash',
 'effort': 'low',
 'where': 'OpenRouter',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'Trip Delay coverage applies under the Premium plan: the flight was canceled by the carrier due to a crew problem and rebooked the next morning, exceeding the 6-hour waiting period. The required carrier notice and receipts are attached. Covered items are hotel $245, meals $95, and local transport $80, totaling $420. No carrier reimbursement or deductible applies to Trip Delay, and $420 is below the $800 maximum.',
 'provider': 'DigitalOcean',
 'input_tokens': 1055,
 'cost': 0.003019185,
 'seconds': 19.0}

### What The Smaller Prompt Saves

In [83]:
def make_day2_messages(claim):
    """ Day 2's prompt: the whole policy wording, and the 
    policy record and the claims as indented JSON
    """
    policy = policies[claim['policy_id']]
    return [
        {
            'role': 'system',
            'content': f"{INSTRUCTIONS}\n{[policy_wording]}\n"
        },
        {
            'role': 'user',
            'content': f"# Policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}\n"
        },
    ]
day2 = openrouter.chat.completions.parse(
    model='deepseek/deepseek-v4.1-flash',
    messages=make_day2_messages(claims[0]),
    response_format=ClaimDecision,
    reasoning_effort='low',
    extra_body={'provider': {'require_parameters': True}},
)

print(f"Day 2's prompt: {day2.usage.prompt_tokens:,} input tokens")
print(f"Today's prompt: {first['input_tokens']:,} input tokens")

Day 2's prompt: 2,808 input tokens
Today's prompt: 1,055 input tokens


### Now Run All 6 Models

In [84]:
def is_right(answer, label):
    """ True when the decision matches the handler's and the payout is within a dollar."""
    return answer['decision'] == label['decision'] and abs(answer['payout'] - label['payout']) <= 1

def score(answer, label):
    """ Add the difficulty, and whether the decision and the payout match the handler's. """
    return {
        **answer,
        'difficulty': label['difficulty'],
        'decision_right': answer['decision'] == label['decision'],
        'right': is_right(answer, label),
    }

def report(results):
    """ Print how a run did: the decisions right, the decisions right with the payout too, the cost and the time. """
    decisions_right = sum(result['decision_right'] for result in results)
    right = sum(result['right'] for result in results)
    cost = sum(result['cost'] for result in results)
    seconds = sum(result['seconds'] for result in results)
    print(f"{results[0]['model']}: {decisions_right} of {len(results)} decisions right, {right} with the payout too, ${cost:.4f} in {seconds:.0f} seconds ")
    

In [85]:
def run(model, effort, claims, labels, ignore=None):
    """ Run a model on OpenRouter on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide(claim, model, effort, ignore)
        results.append(score(answer, labels[claim['claim_id']]))
    report(results)
    return results

In [17]:
mimo = run('xiaomi/mimo-v2.6-pro', None, claims, labels)

xiaomi/mimo-v2.6-pro:   0%|          | 0/20 [00:00<?, ?it/s]

xiaomi/mimo-v2.6-pro: 100%|██████████| 20/20 [05:59<00:00, 17.99s/it]

xiaomi/mimo-v2.6-pro: 16 of 20 decisions right, 13 with the payout too, $0.0237 in 360 seconds 


In [18]:
glm = run('z-ai/glm-5.3-flash', 'low', claims, labels, ignore=['wafer'])

z-ai/glm-5.3-flash: 100%|██████████| 20/20 [02:54<00:00,  8.72s/it]

z-ai/glm-5.3-flash: 15 of 20 decisions right, 11 with the payout too, $0.0035 in 174 seconds 


In [19]:
kimi = run('moonshotai/kimi-k3', 'low', claims, labels)

moonshotai/kimi-k3: 100%|██████████| 20/20 [01:46<00:00,  5.33s/it]

moonshotai/kimi-k3: 17 of 20 decisions right, 16 with the payout too, $0.1347 in 106 seconds 


In [20]:
deepseek = run('deepseek/deepseek-v4.1-flash', 'low', claims, labels)

deepseek/deepseek-v4.1-flash: 100%|██████████| 20/20 [02:39<00:00,  7.95s/it]

deepseek/deepseek-v4.1-flash: 17 of 20 decisions right, 16 with the payout too, $0.0139 in 159 seconds 


In [21]:
nemotron = run('nvidia/nemotron-3-ultra-550b-a55b', 'medium', claims, labels)

nvidia/nemotron-3-ultra-550b-a55b: 100%|██████████| 20/20 [01:59<00:00,  5.99s/it]

nvidia/nemotron-3-ultra-550b-a55b: 16 of 20 decisions right, 13 with the payout too, $0.0223 in 120 seconds 


In [22]:
mistral = run('mistralai/mistral-medium-3-5', 'none', claims, labels)

mistralai/mistral-medium-3-5: 100%|██████████| 20/20 [00:28<00:00,  1.43s/it]

mistralai/mistral-medium-3-5: 13 of 20 decisions right, 11 with the payout too, $0.0531 in 28 seconds 


## Part 3: Three Models On Your Computer With Ollama

In [23]:
print(ollama.pull('qwen3.5:4b').status)
print(ollama.pull('gemma4:e4b').status)
print(ollama.pull('gemma4:e4b').status)

success
success
success


In [86]:
local = ollama.Client(timeout=30) # a single claim should never take more than 2 minutes

In [90]:
import httpx
from pydantic import ValidationError
def decide_locally(claim, model, tries=3):
    """ Ask a model on this computer to decide one claim, through Ollama."""
    start = time.time()
    for attempt in range(tries):
        try:
            response = local.chat(
                model=model,
                messages=make_messages(claim),
                format=LocalDecision.model_json_schema(),
                think=False,
                options={'num_ctx': 8192, 
                         'num_predict': 2048, 
                         'temperature': 0.3 * attempt},
            )
            answer = LocalDecision.model_validate_json(response.message.content)
            break
        except (httpx.ReadTimeout, ValidationError):
            print(f"Failed on {claim['claim_id']} (attempt {attempt + 1})")
            local.generate(model=model, keep_alive=0) # reset a stuck runner; the retry reloads it.
            if attempt == tries -1:
                raise
        
    
    return{
        'model': model,
        'effort': 'none',
        'where': 'This computer',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'input_tokens': response.prompt_eval_count,
        'cost': 0.0,
        'seconds': round(time.time() - start, 1)
    }

In [91]:
def run_locally(model, claims, labels):
    """ Run a model on this computer on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_locally(claim, model)
        results.append(score(answer, labels[claim['claim_id']]))
    ollama.generate(model=model, 
                    keep_alive=0)# unload the model, so the next one has the memory.
    report(results)
    return results

In [55]:
first_local = decide_locally(claims[0], 'qwen3.5:4b')
first_local

{'model': 'qwen3.5:4b',
 'effort': 'none',
 'where': 'This computer',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The claim involves a trip delay where the carrier canceled the flight due to a crew problem (an insured cause). The delay lasted for 24 hours (from June 18 to June 19), which exceeds the Premium plan's Trip Delay waiting period of 6 hours. The claimant is requesting reimbursement for accommodation ($245), meals ($95), and local transport ($80), totaling $420. Under the Payment section, each item is paid up to the lower of the claimed amount or the per-item limit. For the Premium plan, the 'Baggage maximum' table does not provide specific per-item limits for accommodation, meals, or transport; however, standard trip delay clauses allow payment up to the Trip Delay maximum ($800). The claimant's amounts are all reasonable and do not exceed typical daily rates, nor do they clearly violate per-item constraints found in baggage sections (whi

In [93]:
qwen = run_locally('qwen3.5:4b', claims, labels)

qwen3.5:4b:  50%|█████     | 10/20 [01:01<00:49,  4.98s/it]

Failed on CLM-1011 (attempt 1)
Failed on CLM-1011 (attempt 2)


qwen3.5:4b: 100%|██████████| 20/20 [03:17<00:00,  9.86s/it]

qwen3.5:4b: 12 of 20 decisions right, 7 with the payout too, $0.0000 in 197 seconds 


In [92]:
gemma_e4b = run_locally('gemma4:e4b', claims, labels)

gemma4:e4b: 100%|██████████| 20/20 [01:32<00:00,  4.62s/it]

gemma4:e4b: 15 of 20 decisions right, 11 with the payout too, $0.0000 in 92 seconds 


In [45]:
print(ollama.pull('gemma4:e2b-it-qat'))

status='success' completed=None total=None digest=None


In [46]:
gemma_e2b = run_locally('gemma4:e2b-it-qat', claims, labels)

gemma4:e2b-it-qat: 100%|██████████| 20/20 [01:13<00:00,  3.70s/it]

gemma4:e2b-it-qat: 13 of 20 decisions right, 4 with the payout too, $0.0000 in 74 seconds 


In [28]:
cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
CEREBRAS_BASE_URL = "https://api.cerebras.ai/v1"

if cerebras_api_key:
    print("Cerebras API key found")
    cerebras = OpenAI(base_url=CEREBRAS_BASE_URL, api_key=cerebras_api_key, timeout=120, max_retries=5)
else:
    print("No Cerebras API key found. Add CEREBRAS_API_KEY to the .env file")

Cerebras API key found


In [34]:
CEREBRAS_INPUT_PRICE = 0.99 / 1_000_000
CEREBRAS_OUTPUT_PRICE = 1.49 / 1_000_000

def decide_on_cerebras(claim, model, effort):
    """ Ask a model on Cerebras to decide one claim."""
    start = time.time()
    response = cerebras.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
    )
    answer =response.choices[0].message.parsed
    usage = response.usage
    return {
        'model': model,
        'effort': effort,
        'where': 'Cerebras',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'input_tokens': usage.prompt_tokens,
        'cost': usage.prompt_tokens * CEREBRAS_INPUT_PRICE + usage.completion_tokens * CEREBRAS_OUTPUT_PRICE,
        'seconds': round(time.time() - start, 1)
        
    }

def run_on_cerebras(model, effort, claims, labels):
    """ Run a model on Cerebras on the claims and compare against the labels. """
    results =[]
    for claim in tqdm(claims, desc=model):
        answer = decide_on_cerebras(claim, model, effort)
        results.append(score(answer, labels[claim['claim_id']]))
    report(results)
    return results

In [35]:
qwen_cerebras = run_on_cerebras('qwen-3.8-27b','none', claims,labels)

qwen-3.8-27b: 100%|██████████| 20/20 [00:11<00:00,  1.80it/s]

qwen-3.8-27b: 13 of 20 decisions right, 10 with the payout too, $0.0377 in 11 seconds 


In [49]:
qwen_cerebras_low = run_on_cerebras('qwen-3.8-27b',
                                    'low',
                                    claims,
                                    labels)

qwen-3.8-27b: 100%|██████████| 20/20 [00:21<00:00,  1.06s/it]

qwen-3.8-27b: 17 of 20 decisions right, 14 with the payout too, $0.0585 in 21 seconds 


In [50]:
qwen_cerebras_high = run_on_cerebras('qwen-3.8-27b',
                                     'high',
                                     claims,
                                     labels)

qwen-3.8-27b: 100%|██████████| 20/20 [00:56<00:00,  2.82s/it]

qwen-3.8-27b: 16 of 20 decisions right, 13 with the payout too, $0.1030 in 56 seconds 


## Part 4: Accuracy, Cost, and Speed Side by Side

In [94]:
experiments = [mimo,
               glm,
               kimi,
               deepseek,
               nemotron,
               mistral,
               qwen_cerebras,
               qwen_cerebras_low,
               qwen_cerebras_high,
               gemma_e2b,
               gemma_e4b
               ]
rows = []
for results in experiments:
    count = len(results)
    rows.append({
        'model': results[0]['model'].split("/")[-1],
        'where': results[0]['where'],
        'reasoning': results[0]['effort'] or 'not set',
        'decision_right (%)': round(100 * sum(result['decision_right'] for result in results) / count),
        'decision and payout right (%)': round(100 * sum(result['right'] for result in results) / count),
        'seconds a claim': round(sum(result['seconds'] for result in results) / count, 1),
        'cost a claim($)':round(sum(result['cost'] for result in results) / count, 5)
    })

summary = pd.DataFrame(rows)
summary

,model,where,reasoning,decision_right (%),decision and payout right (%),seconds a claim,cost a claim($)
0,mimo-v2.6-pro,OpenRouter,not set,80,65,18.0,0.00118
1,glm-5.3-flash,OpenRouter,low,75,55,8.7,0.00017
2,kimi-k3,OpenRouter,low,85,80,5.3,0.00674
3,deepseek-v4.1-flash,OpenRouter,low,85,80,8.0,0.00069
4,nemotron-3-ultra-550b-a55b,OpenRouter,medium,80,65,6.0,0.00112
5,mistral-medium-3-5,OpenRouter,none,65,55,1.4,0.00266
6,qwen-3.8-27b,Cerebras,none,65,50,0.6,0.00188
7,qwen-3.8-27b,Cerebras,low,85,70,1.0,0.00292
8,qwen-3.8-27b,Cerebras,high,80,65,2.8,0.00515
9,gemma4:e2b-it-qat,This computer,none,65,20,3.7,0.00000


### Accuracy Against Cost

In [99]:
fig = px.scatter(
    summary, 
    x='cost a claim($)',
    y='decision and payout right (%)',
    color='where',
    text='model',
    title='Accuracy vs Cost'
    )
fig.update_traces(marker_size=14, textposition='top center')
fig.update_yaxes(range=[0, 100])
fig.show()

### Accuracy against speed

In [102]:
fig = px.scatter(
    summary,
    x='seconds a claim',
    y='decision and payout right (%)',
    color='where',
    text='model',
    title='Accuracy vs Speed',
    height=600
)
fig.update_traces(marker_size=14, textposition='top right')
fig.update_yaxes(range=[0, 100])
fig.show()


### All Three At Once

In [104]:
fig = px.scatter_3d(
    summary,
    x='cost a claim($)',
    y='seconds a claim',
    z='decision and payout right (%)',
    color='where',
    text='model',
    title='Accuracy vs Cost and Time',
    height=700
)
fig.show()